<a href="https://colab.research.google.com/github/Gk074/NLP-Hate-Speech-Analysis/blob/main/NLP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
reviewer_df = pd.read_csv("/content/reviewer.csv")
misuse_df = pd.read_csv("/content/misuse.csv")

In [3]:
print("reviewer.csv shape:", reviewer_df.shape)
print("misuse.csv shape:", misuse_df.shape)

reviewer.csv shape: (1000, 11)
misuse.csv shape: (1884, 3)


In [4]:
print("\nreviewer.csv columns:")
print(reviewer_df.columns.tolist())

print("\nmisuse.csv columns:")
print(misuse_df.columns.tolist())


reviewer.csv columns:
['Unnamed: 0', 'appID', 'body', 'recordID', 'relevant', 'reviewID', 'score', 'timestamp', 'title', 'Potential or Activity', 'Victim/Abuser/Third']

misuse.csv columns:
['Review', 'Convincingness', 'Severity']


In [5]:
reviewer_df.head()

,Unnamed: 0,appID,body,recordID,relevant,reviewID,score,timestamp,title,Potential or Activity,Victim/Abuser/Third
0,300.0,378450421.0,This app is unlike any other one. It gives you...,580.0,y,3.520784e+08,NaN,2010-09-25T01:42:02Z,Why isn't it in top 10,A,A
1,256.0,378450421.0,Great App. This is what I was looking/waiting ...,631.0,y,3.164352e+08,NaN,2010-07-04T00:31:12Z,Great,P,T
2,107.0,886626229.0,i can spy on my child whenever i want its amaz...,4198.0,y,4.072889e+09,NaN,2019-04-28T17:19:11Z,It's a good app..,A,A
3,30.0,886626229.0,Great app for families keeping track of their ...,4203.0,NaN,5.086004e+09,NaN,2019-11-04T17:48:28Z,Love,A,A
4,278.0,886626229.0,App work great. Another way of spying on people.,4676.0,NaN,3.805983e+09,NaN,2019-02-23T18:36:58Z,amazingggg,A,A


In [6]:
misuse_df.head()

,Review,Convincingness,Severity
0,Great app 4 Stalking.,3.0,3.0
1,I have this on my grandson’s phone. It tells ...,4.0,4.0
2,I read the negative review from a cry baby tee...,4.0,4.0
3,Good app to see your stalkers.,1.5,1.0
4,Love it! Great to stalk family members lol,3.0,3.0


In [7]:
misuse_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1884 entries, 0 to 1883
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Review          1884 non-null   object 
 1   Convincingness  1884 non-null   float64
 2   Severity        1884 non-null   float64
dtypes: float64(2), object(1)
memory usage: 44.3+ KB


In [8]:
reviewer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             402 non-null    float64
 1   appID                  403 non-null    float64
 2   body                   403 non-null    object 
 3   recordID               402 non-null    float64
 4   relevant               91 non-null     object 
 5   reviewID               402 non-null    float64
 6   score                  47 non-null     float64
 7   timestamp              402 non-null    object 
 8   title                  401 non-null    object 
 9   Potential or Activity  403 non-null    object 
 10  Victim/Abuser/Third    403 non-null    object 
dtypes: float64(5), object(6)
memory usage: 86.1+ KB


In [9]:
reviewer_df.isna().sum()

,0
Unnamed: 0,598
appID,597
body,597
recordID,598
relevant,909
reviewID,598
score,953
timestamp,598
title,599
Potential or Activity,597


In [10]:
misuse_df.isna().sum()

,0
Review,0
Convincingness,0
Severity,0


In [11]:
reviewer_df["Victim/Abuser/Third"].value_counts(dropna=False)

,count
Victim/Abuser/Third,
NaN,597
A,223
V,121
T,59


In [12]:
reviewer_df["relevant"].value_counts(dropna=False)

,count
relevant,
NaN,909
y,64
n,27


In [13]:
reviewer_clean = reviewer_df[
    reviewer_df["body"].notna() & reviewer_df["Victim/Abuser/Third"].notna()
].copy()

print("Clean reviewer shape:", reviewer_clean.shape)

Clean reviewer shape: (403, 11)


In [14]:
reviewer_clean[["body", "relevant", "Potential or Activity", "Victim/Abuser/Third"]].head()

,body,relevant,Potential or Activity,Victim/Abuser/Third
0,This app is unlike any other one. It gives you...,y,A,A
1,Great App. This is what I was looking/waiting ...,y,P,T
2,i can spy on my child whenever i want its amaz...,y,A,A
3,Great app for families keeping track of their ...,NaN,A,A
4,App work great. Another way of spying on people.,NaN,A,A


In [15]:
role_map = {
    "A": "Abuser",
    "V": "Victim",
    "T": "Third-party"
}

reviewer_clean["role"] = reviewer_clean["Victim/Abuser/Third"].map(role_map)

reviewer_clean["role"].value_counts()

,count
role,
Abuser,223
Victim,121
Third-party,59


In [16]:
reviewer_clean["relevant"].value_counts(dropna=False)
pd.crosstab(reviewer_clean["role"], reviewer_clean["relevant"], dropna=False)

relevant,n,y,NaN
role,,,
Abuser,19,35,169
Third-party,7,16,36
Victim,1,13,107


In [17]:
import re

In [18]:
keyword_pattern = re.compile(r"\b(spy\w*|stalk\w*|stealth\w*)\b", flags=re.IGNORECASE)

In [19]:
def split_into_sentences(text):
    if pd.isna(text):
        return []
    text = str(text).strip()
    if not text:
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

In [20]:
def extract_keyword_sentences(text):
    sentences = split_into_sentences(text)
    matched = [s for s in sentences if keyword_pattern.search(s)]
    return matched

In [21]:
reviewer_clean["keyword_sentences"] = reviewer_clean["body"].apply(extract_keyword_sentences)
reviewer_clean["num_keyword_sentences"] = reviewer_clean["keyword_sentences"].apply(len)

In [22]:
print("Rows with at least one keyword sentence:",
      (reviewer_clean["num_keyword_sentences"] > 0).sum())

print("Total keyword-matched sentences:",
      reviewer_clean["num_keyword_sentences"].sum())

Rows with at least one keyword sentence: 399
Total keyword-matched sentences: 421


In [23]:
reviewer_clean.loc[
    reviewer_clean["num_keyword_sentences"] > 0,
    ["role", "body", "keyword_sentences"]
].head(10)

,role,body,keyword_sentences
0,Abuser,This app is unlike any other one. It gives you...,[I love that you can take a picture and am con...
1,Third-party,Great App. This is what I was looking/waiting ...,[This is also good to spy on someone.]
2,Abuser,i can spy on my child whenever i want its amaz...,[i can spy on my child whenever i want its ama...
3,Abuser,Great app for families keeping track of their ...,[Great app for families keeping track of their...
4,Abuser,App work great. Another way of spying on people.,[Another way of spying on people.]
5,Abuser,Now that I can spy on my wife I'll always know...,[Now that I can spy on my wife I'll always kno...
6,Abuser,Now that I can spy on my wife I&#39;ll always ...,[Now that I can spy on my wife I&#39;ll always...
7,Abuser,"This app use to work well, now with the latest...",[Now I have to find another app or just pay fo...
8,Third-party,I'm only a kid just want to spy on my sister t...,[I'm only a kid just want to spy on my sister ...
9,Abuser,"Honestly, as a teen, I love this app.\nNow, my...",[Well me and my friends are insane so we mostl...


In [24]:
task1_sentences = reviewer_clean.loc[
    reviewer_clean["num_keyword_sentences"] > 0,
    ["role", "keyword_sentences"]
].explode("keyword_sentences").rename(columns={"keyword_sentences": "sentence"}).reset_index(drop=True)

In [25]:
print(task1_sentences.shape)
task1_sentences.head(10)

(421, 2)


,role,sentence
0,Abuser,I love that you can take a picture and am cons...
1,Third-party,This is also good to spy on someone.
2,Abuser,i can spy on my child whenever i want its amaz...
3,Abuser,Great app for families keeping track of their ...
4,Abuser,Another way of spying on people.
5,Abuser,Now that I can spy on my wife I'll always know...
6,Abuser,Now that I can spy on my wife I&#39;ll always ...
7,Abuser,Now I have to find another app or just pay for...
8,Third-party,I'm only a kid just want to spy on my sister t...
9,Abuser,Well me and my friends are insane so we mostly...


In [26]:
task1_sentences["role"].value_counts()

,count
role,
Abuser,224
Victim,136
Third-party,61


In [27]:
!pip -q install spacy
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [28]:
import spacy
import html

In [29]:
nlp = spacy.load("en_core_web_sm")

In [30]:
def clean_sentence_text(text):
    text = html.unescape(str(text))
    text = text.replace("\n", " ").strip()
    return text

In [31]:
task1_sentences["sentence_clean"] = task1_sentences["sentence"].apply(clean_sentence_text)

In [32]:
def extract_adjectives(text):
    doc = nlp(text)
    adjectives = [token.lemma_.lower() for token in doc if token.pos_ == "ADJ"]
    return adjectives

In [33]:
task1_sentences["adjectives"] = task1_sentences["sentence_clean"].apply(extract_adjectives)
task1_sentences["num_adjectives"] = task1_sentences["adjectives"].apply(len)

In [34]:
print("Total misuse-related sentences:", len(task1_sentences))
print("Sentences with at least one adjective:", (task1_sentences["num_adjectives"] > 0).sum())
print("Total extracted adjectives:", task1_sentences["num_adjectives"].sum())

Total misuse-related sentences: 421
Sentences with at least one adjective: 272
Total extracted adjectives: 443


In [35]:
task1_sentences.loc[
    task1_sentences["num_adjectives"] > 0,
    ["role", "sentence_clean", "adjectives"]
].head(15)

,role,sentence_clean,adjectives
1,Third-party,This is also good to spy on someone.,[good]
2,Abuser,i can spy on my child whenever i want its amaz...,[amazing]
3,Abuser,Great app for families keeping track of their ...,"[great, creepy]"
7,Abuser,Now I have to find another app or just pay for...,[sure]
8,Third-party,I'm only a kid just want to spy on my sister t...,[good]
9,Abuser,Well me and my friends are insane so we mostly...,"[insane, other]"
13,Victim,Teen years is a time to have fun and with ur p...,"[teen, bad]"
14,Abuser,This app is useful to stalk my parents but hon...,[useful]
16,Victim,absolutely horrible y’all all about STALKING Y...,[horrible]
17,Victim,as a parent i believe that parents need to hav...,"[real, worthy]"


In [36]:
task1_sentences.groupby("role")["num_adjectives"].sum()

,num_adjectives
role,
Abuser,212
Third-party,79
Victim,152


In [37]:
task1_adj = task1_sentences.loc[
    task1_sentences["num_adjectives"] > 0,
    ["role", "sentence_clean", "adjectives"]
].explode("adjectives").rename(columns={"adjectives": "adjective"}).reset_index(drop=True)

In [38]:
task1_adj["adjective"] = task1_adj["adjective"].astype(str).str.strip().str.lower()
task1_adj = task1_adj[task1_adj["adjective"] != ""].copy()

In [39]:
print("Total adjective tokens:", len(task1_adj))
print("Unique adjectives:", task1_adj["adjective"].nunique())
task1_adj.head(10)

Total adjective tokens: 443
Unique adjectives: 173


,role,sentence_clean,adjective
0,Third-party,This is also good to spy on someone.,good
1,Abuser,i can spy on my child whenever i want its amaz...,amazing
2,Abuser,Great app for families keeping track of their ...,great
3,Abuser,Great app for families keeping track of their ...,creepy
4,Abuser,Now I have to find another app or just pay for...,sure
5,Third-party,I'm only a kid just want to spy on my sister t...,good
6,Abuser,Well me and my friends are insane so we mostly...,insane
7,Abuser,Well me and my friends are insane so we mostly...,other
8,Victim,Teen years is a time to have fun and with ur p...,teen
9,Victim,Teen years is a time to have fun and with ur p...,bad


In [40]:
task1_adj["adjective"].value_counts().head(20)

,count
adjective,
good,45
great,34
other,22
cool,12
crazy,11
new,10
easy,9
bad,9
safe,9


In [41]:
for role_name in task1_adj["role"].unique():
    print(f"\nTop adjectives for {role_name}:")
    print(task1_adj.loc[task1_adj["role"] == role_name, "adjective"].value_counts().head(15))


Top adjectives for Third-party:
adjective
good           10
other           4
real            3
great           3
new             3
annoying        2
young           2
crazy           2
weird           2
legalized       1
disgusting      1
low             1
disgraceful     1
own             1
sure            1
Name: count, dtype: int64

Top adjectives for Abuser:
adjective
good       26
great      24
other      11
cool       11
easy        8
perfect     8
awesome     5
whole       5
useful      5
nice        5
young       4
same        4
safe        3
creepy      3
sure        3
Name: count, dtype: int64

Top adjectives for Victim:
adjective
good          9
bad           8
other         7
great         7
crazy         7
new           5
safe          5
nosy          4
little        3
stalking      2
only          2
free          2
disgusting    2
normal        2
ok            2
Name: count, dtype: int64


In [42]:
task1_adj["role"].value_counts()

,count
role,
Abuser,212
Victim,152
Third-party,79


In [43]:
!wget -O /content/NRC-VAD-Lexicon.zip "http://saifmohammad.com/WebDocs/Lexicons/NRC-VAD-Lexicon.zip"
!unzip -o /content/NRC-VAD-Lexicon.zip -d /content/nrc_vad

--2026-04-21 22:18:13--  http://saifmohammad.com/WebDocs/Lexicons/NRC-VAD-Lexicon.zip
Resolving saifmohammad.com (saifmohammad.com)... 192.185.17.122
Connecting to saifmohammad.com (saifmohammad.com)|192.185.17.122|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 44536009 (42M) [application/zip]
Saving to: ‘/content/NRC-VAD-Lexicon.zip’

/content/NRC-VAD-Le 100%[===================>]  42.47M  8.59MB/s    in 6.4s    

2026-04-21 22:18:20 (6.62 MB/s) - ‘/content/NRC-VAD-Lexicon.zip’ saved [44536009/44536009]

Archive:  /content/NRC-VAD-Lexicon.zip
  inflating: /content/nrc_vad/NRC-VAD-Lexicon/NRC-VAD-Lexicon-ForVariousLanguages.txt  
  inflating: /content/nrc_vad/NRC-VAD-Lexicon/Paper-Practical-Ethical-Considerations-Lexicons.pdf  
  inflating: /content/nrc_vad/__MACOSX/NRC-VAD-Lexicon/._Paper-Practical-Ethical-Considerations-Lexicons.pdf  
  inflating: /content/nrc_vad/NRC-VAD-Lexicon/Paper-VAD-ACL2018.pdf  
  inflating: /content/nrc_vad/__MACOSX/NRC-VAD-Lexicon/

In [44]:
import os

for root, dirs, files in os.walk("/content/nrc_vad"):
    for f in files:
        print(os.path.join(root, f))

/content/nrc_vad/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/ListOfLanguages-For-Which-Lexicon-Availabale.txt
/content/nrc_vad/NRC-VAD-Lexicon/Paper-Practical-Ethical-Considerations-Lexicons.pdf
/content/nrc_vad/NRC-VAD-Lexicon/README.txt
/content/nrc_vad/NRC-VAD-Lexicon/Paper-Ethics-Sheet-Emotion-Recognition.pdf
/content/nrc_vad/NRC-VAD-Lexicon/NRC-VAD-Lexicon-ForVariousLanguages.txt
/content/nrc_vad/NRC-VAD-Lexicon/Paper-VAD-ACL2018.pdf
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Zulu-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Dutch-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Shona-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Lao-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Latvian-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Mongolian-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Bengali-N

In [45]:
import os
import pandas as pd

In [46]:
for root, dirs, files in os.walk("/content/nrc_vad"):
    for f in files:
        print(os.path.join(root, f))

/content/nrc_vad/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/ListOfLanguages-For-Which-Lexicon-Availabale.txt
/content/nrc_vad/NRC-VAD-Lexicon/Paper-Practical-Ethical-Considerations-Lexicons.pdf
/content/nrc_vad/NRC-VAD-Lexicon/README.txt
/content/nrc_vad/NRC-VAD-Lexicon/Paper-Ethics-Sheet-Emotion-Recognition.pdf
/content/nrc_vad/NRC-VAD-Lexicon/NRC-VAD-Lexicon-ForVariousLanguages.txt
/content/nrc_vad/NRC-VAD-Lexicon/Paper-VAD-ACL2018.pdf
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Zulu-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Dutch-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Shona-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Lao-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Latvian-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Mongolian-NRC-VAD-Lexicon.txt
/content/nrc_vad/NRC-VAD-Lexicon/OneFilePerLanguage/Bengali-N

In [47]:
vad_path = "/content/nrc_vad/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt"
vad_df = pd.read_csv(
    vad_path,
    sep="\t",
    header=None,
    names=["Word", "Valence", "Arousal", "Dominance"]
)
vad_df.head()

,Word,Valence,Arousal,Dominance
0,aaaaaaah,0.479,0.606,0.291
1,aaaah,0.520,0.636,0.282
2,aardvark,0.427,0.490,0.437
3,aback,0.385,0.407,0.288
4,abacus,0.510,0.276,0.485


In [48]:
vad_df["Word"] = vad_df["Word"].astype(str).str.strip().str.lower()

task1_vad = task1_adj.merge(
    vad_df,
    left_on="adjective",
    right_on="Word",
    how="inner"
)

task1_role_vad = task1_vad.groupby("role")[["Valence", "Arousal", "Dominance"]].mean().round(4)
task1_role_vad

,Valence,Arousal,Dominance
role,,,
Abuser,0.7428,0.4810,0.6019
Third-party,0.5945,0.4708,0.5112
Victim,0.5394,0.5024,0.4976


In [49]:
task1_vad["role"].value_counts()

,count
role,
Abuser,180
Victim,124
Third-party,66


In [50]:
for role_name in ["Abuser", "Victim", "Third-party"]:
    print(f"\nTop matched adjectives for {role_name}:")
    print(
        task1_vad.loc[task1_vad["role"] == role_name, "adjective"]
        .value_counts()
        .head(15)
    )


Top matched adjectives for Abuser:
adjective
good       26
great      24
other      11
cool       11
perfect     8
easy        8
awesome     5
whole       5
useful      5
nice        5
creepy      3
sure        3
safe        3
little      3
able        2
Name: count, dtype: int64

Top matched adjectives for Victim:
adjective
good      9
bad       8
other     7
great     7
crazy     7
new       5
safe      5
nosy      4
little    3
normal    2
super     2
full      2
real      2
sure      2
free      2
Name: count, dtype: int64

Top matched adjectives for Third-party:
adjective
good         10
other         4
new           3
real          3
great         3
annoying      2
crazy         2
weird         2
close         1
different     1
cool          1
intrusive     1
cold          1
simple        1
sure          1
Name: count, dtype: int64


In [51]:
task1_role_vad.reset_index()

,role,Valence,Arousal,Dominance
0,Abuser,0.7428,0.4810,0.6019
1,Third-party,0.5945,0.4708,0.5112
2,Victim,0.5394,0.5024,0.4976


In [52]:
task1_role_vad.reset_index().to_csv("/content/task1_role_vad_results.csv", index=False)

In [53]:
misuse_df[["Convincingness", "Severity"]].describe()
misuse_df[["Convincingness", "Severity"]].nunique()
misuse_df.head()

,Review,Convincingness,Severity
0,Great app 4 Stalking.,3.0,3.0
1,I have this on my grandson’s phone. It tells ...,4.0,4.0
2,I read the negative review from a cry baby tee...,4.0,4.0
3,Good app to see your stalkers.,1.5,1.0
4,Love it! Great to stalk family members lol,3.0,3.0


In [54]:
task2_df = misuse_df.copy()

sev_threshold = task2_df["Severity"].median()
conv_threshold = task2_df["Convincingness"].median()

task2_df["sev_cat"] = (task2_df["Severity"] >= sev_threshold).astype(int)
task2_df["conv_cat"] = (task2_df["Convincingness"] >= conv_threshold).astype(int)

task2_df["joint_label_name"] = (
    task2_df["sev_cat"].map({0: "SevLow", 1: "SevHigh"}) + "_" +
    task2_df["conv_cat"].map({0: "ConvLow", 1: "ConvHigh"})
)

task2_df = task2_df.rename(columns={"Review": "text"})

In [55]:
print("Severity median:", sev_threshold)
print("Convincingness median:", conv_threshold)

print("\nSeverity class counts:")
print(task2_df["sev_cat"].value_counts())

print("\nConvincingness class counts:")
print(task2_df["conv_cat"].value_counts())

print("\nJoint label counts:")
print(task2_df["joint_label_name"].value_counts())

Severity median: 1.0
Convincingness median: 1.0

Severity class counts:
sev_cat
1    1884
Name: count, dtype: int64

Convincingness class counts:
conv_cat
1    1884
Name: count, dtype: int64

Joint label counts:
joint_label_name
SevHigh_ConvHigh    1884
Name: count, dtype: int64


In [56]:
label_map = {label: idx for idx, label in enumerate(sorted(task2_df["joint_label_name"].unique()))}
id_to_label = {v: k for k, v in label_map.items()}

task2_df["labels"] = task2_df["joint_label_name"].map(label_map)

task2_df[["text", "Severity", "Convincingness", "joint_label_name", "labels"]].head()

,text,Severity,Convincingness,joint_label_name,labels
0,Great app 4 Stalking.,3.0,3.0,SevHigh_ConvHigh,0
1,I have this on my grandson’s phone. It tells ...,4.0,4.0,SevHigh_ConvHigh,0
2,I read the negative review from a cry baby tee...,4.0,4.0,SevHigh_ConvHigh,0
3,Good app to see your stalkers.,1.0,1.5,SevHigh_ConvHigh,0
4,Love it! Great to stalk family members lol,3.0,3.0,SevHigh_ConvHigh,0


In [57]:
!pip -q install simpletransformers

In [58]:
%pip uninstall -y transformers tokenizers accelerate datasets -q
%pip install -U "transformers[torch]" datasets accelerate scikit-learn --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 368.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 424.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 179.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 360.1 MB/s eta 0:00:00


In [59]:
import transformers, datasets, accelerate, sklearn, torch
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)

transformers: 5.5.4
datasets: 4.8.4
accelerate: 1.13.0
torch: 2.10.0+cu128


In [60]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [61]:
task2_df = misuse_df.copy()

task2_df["text"] = task2_df["Review"].astype(str)

sev_threshold = task2_df["Severity"].median()
conv_threshold = task2_df["Convincingness"].median()

task2_df["sev_cat"] = (task2_df["Severity"] >= sev_threshold).astype(int)
task2_df["conv_cat"] = (task2_df["Convincingness"] >= conv_threshold).astype(int)

task2_df["joint_label_name"] = (
    task2_df["sev_cat"].map({0: "SevLow", 1: "SevHigh"}) + "_" +
    task2_df["conv_cat"].map({0: "ConvLow", 1: "ConvHigh"})
)

label_map = {label: idx for idx, label in enumerate(sorted(task2_df["joint_label_name"].unique()))}
id_to_label = {v: k for k, v in label_map.items()}

task2_df["labels"] = task2_df["joint_label_name"].map(label_map)

print("Severity median:", sev_threshold)
print("Convincingness median:", conv_threshold)
print()
print(task2_df["joint_label_name"].value_counts())

Severity median: 1.0
Convincingness median: 1.0

joint_label_name
SevHigh_ConvHigh    1884
Name: count, dtype: int64


In [62]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [67]:
task2_df["labels"] = task2_df["labels"].astype(int)
task2_df["sev_cat"] = task2_df["sev_cat"].astype(int)
task2_df["conv_cat"] = task2_df["conv_cat"].astype(int)

In [68]:
def decode_joint_label(label_name):
    sev = 1 if "SevHigh" in label_name else 0
    conv = 1 if "ConvHigh" in label_name else 0
    return sev, conv

In [69]:
def make_hf_dataset(texts, labels, tokenizer, max_length=128):
    ds = Dataset.from_dict({
        "text": texts.tolist(),
        "labels": [int(x) for x in labels.tolist()]
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    ds = ds.map(tokenize_batch, batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [81]:
task2_df["text_norm"] = task2_df["text"].astype(str).str.strip().str.lower()
task2_df = task2_df.drop_duplicates(subset="text_norm").copy()

sev_threshold = task2_df["Severity"].median()
conv_threshold = task2_df["Convincingness"].median()

task2_df["sev_cat"] = (task2_df["Severity"] > sev_threshold).astype(int)
task2_df["conv_cat"] = (task2_df["Convincingness"] > conv_threshold).astype(int)

task2_df["joint_label_name"] = (
    task2_df["sev_cat"].map({0: "SevLow", 1: "SevHigh"}) + "_" +
    task2_df["conv_cat"].map({0: "ConvLow", 1: "ConvHigh"})
)

label_map = {label: idx for idx, label in enumerate(sorted(task2_df["joint_label_name"].unique()))}
id_to_label = {v: k for k, v in label_map.items()}
task2_df["labels"] = task2_df["joint_label_name"].map(label_map).astype(int)

print(task2_df["joint_label_name"].value_counts())

joint_label_name
SevLow_ConvLow      1096
SevHigh_ConvHigh     715
SevLow_ConvHigh       66
SevHigh_ConvLow        1
Name: count, dtype: int64


In [83]:
def run_transformer_cv(model_name, df, label_map, id_to_label, num_epochs=1):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df["text"], df["labels"]), start=1):
        print(f"\nRunning {model_name} | Fold {fold}")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        train_ds = make_hf_dataset(train_df["text"], train_df["labels"], tokenizer)
        test_ds = make_hf_dataset(test_df["text"], test_df["labels"], tokenizer)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(label_map)
        )
        model.config.problem_type = "single_label_classification"

        args = TrainingArguments(
            output_dir=f"/content/{model_name.replace('/', '_')}_fold_{fold}",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=num_epochs,
            save_strategy="no",
            logging_strategy="no",
            report_to="none",
            seed=42
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds
        )

        trainer.train()

        pred_output = trainer.predict(test_ds)
        pred_ids = np.argmax(pred_output.predictions, axis=1)
        pred_joint = [id_to_label[int(x)] for x in pred_ids]

        pred_sev, pred_conv = [], []
        for name in pred_joint:
            sev, conv = decode_joint_label(name)
            pred_sev.append(sev)
            pred_conv.append(conv)

        true_sev = test_df["sev_cat"].tolist()
        true_conv = test_df["conv_cat"].tolist()
        true_joint = test_df["joint_label_name"].tolist()

        sev_p, sev_r, sev_f1, _ = precision_recall_fscore_support(
            true_sev, pred_sev, average="binary", zero_division=0
        )
        conv_p, conv_r, conv_f1, _ = precision_recall_fscore_support(
            true_conv, pred_conv, average="binary", zero_division=0
        )
        joint_p, joint_r, joint_f1, _ = precision_recall_fscore_support(
            true_joint, pred_joint, average="macro", zero_division=0
        )

        fold_rows.append({
            "fold": fold,
            "sev_precision": sev_p,
            "sev_recall": sev_r,
            "sev_f1": sev_f1,
            "conv_precision": conv_p,
            "conv_recall": conv_r,
            "conv_f1": conv_f1,
            "joint_macro_precision": joint_p,
            "joint_macro_recall": joint_r,
            "joint_macro_f1": joint_f1
        })

        del model, trainer, train_ds, test_ds, pred_output
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_df = pd.DataFrame(fold_rows)

    summary_df = pd.DataFrame({
        "metric": [
            "Severity Precision",
            "Severity Recall",
            "Severity F1",
            "Convincingness Precision",
            "Convincingness Recall",
            "Convincingness F1",
            "Joint Macro Precision",
            "Joint Macro Recall",
            "Joint Macro F1"
        ],
        "mean": [
            fold_df["sev_precision"].mean(),
            fold_df["sev_recall"].mean(),
            fold_df["sev_f1"].mean(),
            fold_df["conv_precision"].mean(),
            fold_df["conv_recall"].mean(),
            fold_df["conv_f1"].mean(),
            fold_df["joint_macro_precision"].mean(),
            fold_df["joint_macro_recall"].mean(),
            fold_df["joint_macro_f1"].mean()
        ],
        "std": [
            fold_df["sev_precision"].std(),
            fold_df["sev_recall"].std(),
            fold_df["sev_f1"].std(),
            fold_df["conv_precision"].std(),
            fold_df["conv_recall"].std(),
            fold_df["conv_f1"].std(),
            fold_df["joint_macro_precision"].std(),
            fold_df["joint_macro_recall"].std(),
            fold_df["joint_macro_f1"].std()
        ]
    })

    return fold_df, summary_df

In [84]:
distil_folds, distil_summary = run_transformer_cv(
    model_name="distilbert-base-uncased",
    df=task2_df,
    label_map=label_map,
    id_to_label=id_to_label,
    num_epochs=1
)

distil_summary

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



Running distilbert-base-uncased | Fold 1


Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Map:   0%|          | 0/375 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


,metric,mean,std
0,Severity Precision,0.802153,0.033399
1,Severity Recall,0.944134,0.024725
2,Severity F1,0.866876,0.020354
3,Convincingness Precision,0.876844,0.034687
4,Convincingness Recall,0.946219,0.025083
5,Convincingness F1,0.909753,0.021034
6,Joint Macro Precision,0.556858,0.060219
7,Joint Macro Recall,0.584898,0.063579
8,Joint Macro F1,0.568440,0.061053


In [85]:
roberta_folds, roberta_summary = run_transformer_cv(
    model_name="roberta-base",
    df=task2_df,
    label_map=label_map,
    id_to_label=id_to_label,
    num_epochs=1
)

roberta_summary

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



Running roberta-base | Fold 1


Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Map:   0%|          | 0/375 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


,metric,mean,std
0,Severity Precision,0.796650,0.036262
1,Severity Recall,0.959499,0.010363
2,Severity F1,0.870018,0.018824
3,Convincingness Precision,0.868441,0.037006
4,Convincingness Recall,0.959023,0.011659
5,Convincingness F1,0.911017,0.017877
6,Joint Macro Precision,0.557989,0.060591
7,Joint Macro Recall,0.586904,0.063717
8,Joint Macro F1,0.569218,0.061256


In [86]:
distil_summary["model"] = "DistilBERT"
roberta_summary["model"] = "RoBERTa"

task2_comparison = pd.concat([distil_summary, roberta_summary], ignore_index=True)
task2_comparison

,metric,mean,std,model
0,Severity Precision,0.802153,0.033399,DistilBERT
1,Severity Recall,0.944134,0.024725,DistilBERT
2,Severity F1,0.866876,0.020354,DistilBERT
3,Convincingness Precision,0.876844,0.034687,DistilBERT
4,Convincingness Recall,0.946219,0.025083,DistilBERT
5,Convincingness F1,0.909753,0.021034,DistilBERT
6,Joint Macro Precision,0.556858,0.060219,DistilBERT
7,Joint Macro Recall,0.584898,0.063579,DistilBERT
8,Joint Macro F1,0.568440,0.061053,DistilBERT
9,Severity Precision,0.796650,0.036262,RoBERTa


Saving whatever done so far to not rerun the entire models again and again


In [87]:
from pathlib import Path
import shutil

out_dir = Path("/content/project_outputs")
out_dir.mkdir(exist_ok=True)

if "reviewer_clean" in globals():
    reviewer_clean.to_csv(out_dir / "reviewer_clean.csv", index=False)

if "task1_role_vad" in globals():
    task1_role_vad.reset_index().to_csv(out_dir / "task1_role_vad_results.csv", index=False)

if "task2_df" in globals():
    task2_df.to_csv(out_dir / "task2_modeling_data.csv", index=False)

if "distil_folds" in globals():
    distil_folds.to_csv(out_dir / "task2_distil_folds.csv", index=False)

if "distil_summary" in globals():
    distil_summary.to_csv(out_dir / "task2_distil_summary.csv", index=False)

if "roberta_folds" in globals():
    roberta_folds.to_csv(out_dir / "task2_roberta_folds.csv", index=False)

if "roberta_summary" in globals():
    roberta_summary.to_csv(out_dir / "task2_roberta_summary.csv", index=False)

if "task2_comparison" in globals():
    task2_comparison.to_csv(out_dir / "task2_comparison.csv", index=False)

print("Saved files:")
for p in out_dir.iterdir():
    print(p.name)

Saved files:
task2_comparison.csv
task2_distil_summary.csv
task2_roberta_summary.csv
task1_role_vad_results.csv
task2_modeling_data.csv
task2_distil_folds.csv
task2_roberta_folds.csv
reviewer_clean.csv


In [88]:
from sklearn.model_selection import StratifiedShuffleSplit

task3_df = reviewer_clean[["body", "role"]].copy().reset_index(drop=True)
task3_df["sample_id"] = range(len(task3_df))

splitter = StratifiedShuffleSplit(n_splits=1, test_size=200, random_state=42)
source_idx, test_idx = next(splitter.split(task3_df["body"], task3_df["role"]))

source_df = task3_df.iloc[source_idx].reset_index(drop=True)
test_df = task3_df.iloc[test_idx].reset_index(drop=True)

print("Total:", len(task3_df))
print("Source:", len(source_df))
print("Test:", len(test_df))

print("\nSource role counts:")
print(source_df["role"].value_counts())

print("\nTest role counts:")
print(test_df["role"].value_counts())

Total: 403
Source: 203
Test: 200

Source role counts:
role
Abuser         112
Victim          61
Third-party     30
Name: count, dtype: int64

Test role counts:
role
Abuser         111
Victim          60
Third-party     29
Name: count, dtype: int64


In [90]:
source_df.to_csv("/content/project_outputs/task3_source_df.csv", index=False)
test_df.to_csv("/content/project_outputs/task3_test_df.csv", index=False)

In [91]:
zip_path = shutil.make_archive("/content/project_outputs_backup", "zip", "/content/project_outputs")
print(zip_path)

/content/project_outputs_backup.zip


'''''''''''' Task 3 ''''''''''''''''''

In [1]:
!pip -q install -U openai scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.1 MB/s eta 0:00:00


In [2]:
import os
import re
import html
import time
import json
import zipfile
import shutil
from datetime import datetime

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
zip_path = "/content/project_outputs_backup.zip"
extract_dir = "/content/project_outputs"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print(sorted(os.listdir(extract_dir)))

['reviewer_clean.csv', 'task1_role_vad_results.csv', 'task2_comparison.csv', 'task2_distil_folds.csv', 'task2_distil_summary.csv', 'task2_modeling_data.csv', 'task2_roberta_folds.csv', 'task2_roberta_summary.csv', 'task3_source_df.csv', 'task3_test_df.csv']


In [5]:
source_df = pd.read_csv("/content/project_outputs/task3_source_df.csv")
test_df = pd.read_csv("/content/project_outputs/task3_test_df.csv")

print("source shape:", source_df.shape)
print("test shape:", test_df.shape)

print("\nsource role counts")
print(source_df["role"].value_counts())

print("\ntest role counts")
print(test_df["role"].value_counts())

source shape: (203, 3)
test shape: (200, 3)

source role counts
role
Abuser         112
Victim          61
Third-party     30
Name: count, dtype: int64

test role counts
role
Abuser         111
Victim          60
Third-party     29
Name: count, dtype: int64


In [6]:
from google.colab import userdata

os.environ["CEREBRAS_API_KEY"] = userdata.get("CEREBRAS_API_KEY")

client = OpenAI(
    base_url="https://api.cerebras.ai/v1",
    api_key=os.environ["CEREBRAS_API_KEY"]
)

MODEL_NAME = "llama3.1-8b"
print("Cerebras ready")
print("Model:", MODEL_NAME)

Cerebras ready
Model: llama3.1-8b


In [7]:
LABELS = ["Abuser", "Victim", "Third-party"]

def clean_review_text(text):
    text = html.unescape(str(text))
    text = text.replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

source_df["body_clean"] = source_df["body"].apply(clean_review_text)
test_df["body_clean"] = test_df["body"].apply(clean_review_text)

In [8]:
ROLE_DEFINITIONS = """
Label definitions:
- Abuser: the narrator supports, endorses, or personally uses spying, stalking, tracking, or surveillance on someone else.
- Victim: the narrator is being watched, tracked, stalked, controlled, harmed, or speaks from the targeted or privacy-invaded perspective.
- Third-party: the narrator comments on, evaluates, or observes the behavior without clearly being the abuser or the victim.
""".strip()

In [9]:
def build_zero_shot_prompt(review_text, cot=False):
    if cot:
        return f"""
Classify the narrator role in this app review.

{ROLE_DEFINITIONS}

Review:
\"\"\"{review_text}\"\"\"

Think briefly, then return exactly in this format:
Reason: <one short sentence>
Label: <Abuser or Victim or Third-party>
""".strip()
    else:
        return f"""
Classify the narrator role in this app review.

{ROLE_DEFINITIONS}

Review:
\"\"\"{review_text}\"\"\"

Return exactly one label only:
Abuser
Victim
Third-party
""".strip()

In [10]:
def format_examples(examples_df):
    blocks = []
    for i, row in enumerate(examples_df.itertuples(index=False), start=1):
        blocks.append(f"Example {i}\nReview: {row.body_clean}\nLabel: {row.role}")
    return "\n\n".join(blocks)

In [11]:
def build_few_shot_prompt(review_text, examples_df, cot=False):
    examples_block = format_examples(examples_df)

    if cot:
        return f"""
Classify the narrator role in this app review.

{ROLE_DEFINITIONS}

Here are labeled examples:

{examples_block}

Now classify the next review.

Review:
\"\"\"{review_text}\"\"\"

Think briefly, then return exactly in this format:
Reason: <one short sentence>
Label: <Abuser or Victim or Third-party>
""".strip()
    else:
        return f"""
Classify the narrator role in this app review.

{ROLE_DEFINITIONS}

Here are labeled examples:

{examples_block}

Now classify the next review.

Review:
\"\"\"{review_text}\"\"\"

Return exactly one label only:
Abuser
Victim
Third-party
""".strip()

In [12]:
def parse_label(raw_text):
    if raw_text is None:
        return None

    text = str(raw_text).strip().lower()

    if "third-party" in text or "third party" in text:
        return "Third-party"
    if "victim" in text:
        return "Victim"
    if "abuser" in text:
        return "Abuser"

    return None

In [13]:
def call_cerebras(prompt, max_retries=6):
    wait_time = 5

    for _ in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_completion_tokens=16
            )
            return response.choices[0].message.content

        except Exception as e:
            err_text = str(e).lower()
            if "429" in err_text or "rate" in err_text or "limit" in err_text:
                print(f"Rate limited. Sleeping for {wait_time} seconds...")
                time.sleep(wait_time)
                wait_time *= 2
            else:
                raise e

    raise RuntimeError("Max retries exceeded")

In [14]:
results_dir = "/content/task3_results"
meta_dir = os.path.join(results_dir, "meta")
eval_dir = os.path.join(results_dir, "eval")

os.makedirs(results_dir, exist_ok=True)
os.makedirs(meta_dir, exist_ok=True)
os.makedirs(eval_dir, exist_ok=True)

registry_path = os.path.join(results_dir, "task3_setup_registry.csv")

In [15]:
def save_setup_metadata(setup_name, model_name, cot, max_rows, notes=None):
    meta = {
        "setup_name": setup_name,
        "model_name": model_name,
        "cot": cot,
        "max_rows": max_rows,
        "created_at": datetime.utcnow().isoformat(),
        "notes": notes
    }

    meta_path = os.path.join(meta_dir, f"{setup_name}_meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    return meta_path

In [16]:
def update_registry(setup_name, result_df, model_name, cot):
    completed = result_df["pred_label"].notna().sum()
    valid = result_df["pred_label"].isin(LABELS).sum()
    errors = result_df["error"].notna().sum()

    row = pd.DataFrame([{
        "setup_name": setup_name,
        "model_name": model_name,
        "cot": cot,
        "rows_total": len(result_df),
        "rows_completed": completed,
        "valid_predictions": valid,
        "errors": errors,
        "last_updated": datetime.utcnow().isoformat()
    }])

    if os.path.exists(registry_path):
        old = pd.read_csv(registry_path)
        old = old[old["setup_name"] != setup_name]
        new_df = pd.concat([old, row], ignore_index=True)
    else:
        new_df = row

    new_df.to_csv(registry_path, index=False)

In [17]:
def backup_task3_outputs():
    zip_path = shutil.make_archive("/content/task3_results_backup", "zip", results_dir)
    print("Backup written to:", zip_path)
    return zip_path

In [18]:
def run_prompting_setup(setup_name, prompt_builder_fn, cot=False, max_rows=None, save_every=5, notes=None):
    out_path = os.path.join(results_dir, f"{setup_name}.csv")

    save_setup_metadata(
        setup_name=setup_name,
        model_name=MODEL_NAME,
        cot=cot,
        max_rows=max_rows,
        notes=notes
    )

    if os.path.exists(out_path):
        existing = pd.read_csv(out_path)
        done_ids = set(existing["sample_id"].tolist())
        rows = existing.to_dict("records")
        print(f"Resuming {setup_name}. Already completed: {len(done_ids)}")
    else:
        done_ids = set()
        rows = []

    run_df = test_df.copy()
    if max_rows is not None:
        run_df = run_df.iloc[:max_rows].copy()

    pending_counter = 0

    for pos, row in enumerate(tqdm(run_df.itertuples(index=False), total=len(run_df), desc=setup_name)):
        if row.sample_id in done_ids:
            continue

        try:
            prompt = prompt_builder_fn(row, pos, cot)
            raw_output = call_cerebras(prompt)
            pred_label = parse_label(raw_output)
            error_msg = None
        except Exception as e:
            raw_output = None
            pred_label = None
            error_msg = str(e)

        rows.append({
            "sample_id": row.sample_id,
            "true_label": row.role,
            "pred_label": pred_label,
            "raw_output": raw_output,
            "error": error_msg
        })

        pending_counter += 1

        if pending_counter >= save_every:
            tmp_df = pd.DataFrame(rows)
            tmp_df.to_csv(out_path, index=False)
            update_registry(setup_name, tmp_df, MODEL_NAME, cot)
            pending_counter = 0

    result_df = pd.DataFrame(rows)
    result_df.to_csv(out_path, index=False)
    update_registry(setup_name, result_df, MODEL_NAME, cot)

    return result_df

In [19]:
def evaluate_predictions(result_df, setup_name):
    df = result_df[result_df["pred_label"].isin(LABELS)].copy()

    if len(df) == 0:
        eval_df = pd.DataFrame([
            {"setup": setup_name, "label": "Abuser", "precision": 0.0, "recall": 0.0, "f1": 0.0, "support": 0},
            {"setup": setup_name, "label": "Victim", "precision": 0.0, "recall": 0.0, "f1": 0.0, "support": 0},
            {"setup": setup_name, "label": "Third-party", "precision": 0.0, "recall": 0.0, "f1": 0.0, "support": 0},
            {"setup": setup_name, "label": "macro avg", "precision": 0.0, "recall": 0.0, "f1": 0.0, "support": 0},
            {"setup": setup_name, "label": "accuracy", "precision": 0.0, "recall": 0.0, "f1": 0.0, "support": 0}
        ])
    else:
        report = classification_report(
            df["true_label"],
            df["pred_label"],
            labels=LABELS,
            output_dict=True,
            zero_division=0
        )

        rows = []
        for label in LABELS:
            rows.append({
                "setup": setup_name,
                "label": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1": report[label]["f1-score"],
                "support": report[label]["support"]
            })

        rows.append({
            "setup": setup_name,
            "label": "macro avg",
            "precision": report["macro avg"]["precision"],
            "recall": report["macro avg"]["recall"],
            "f1": report["macro avg"]["f1-score"],
            "support": report["macro avg"]["support"]
        })

        rows.append({
            "setup": setup_name,
            "label": "accuracy",
            "precision": report["accuracy"],
            "recall": report["accuracy"],
            "f1": report["accuracy"],
            "support": len(df)
        })

        eval_df = pd.DataFrame(rows)

    eval_df.to_csv(os.path.join(eval_dir, f"{setup_name}_eval.csv"), index=False)
    return eval_df

In [20]:
def run_and_save_setup(setup_name, prompt_builder_fn, cot=False, max_rows=None, save_every=5, notes=None):
    result_df = run_prompting_setup(
        setup_name=setup_name,
        prompt_builder_fn=prompt_builder_fn,
        cot=cot,
        max_rows=max_rows,
        save_every=save_every,
        notes=notes
    )

    eval_df = evaluate_predictions(result_df, setup_name)
    backup_task3_outputs()

    return result_df, eval_df

In [21]:
def zero_shot_builder(row, pos, cot=False):
    return build_zero_shot_prompt(row.body_clean, cot=cot)

In [38]:
zero_shot_test10, zero_shot_test10_eval = run_and_save_setup(
    setup_name="zero_shot_test10_cerebras",
    prompt_builder_fn=zero_shot_builder,
    cot=False,
    max_rows=10,
    save_every=2,
    notes="Task 3 zero-shot sanity check"
)

zero_shot_test10[["sample_id", "true_label", "pred_label", "raw_output", "error"]]

/tmp/ipykernel_2159/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


zero_shot_test10_cerebras:   0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


,sample_id,true_label,pred_label,raw_output,error
0,357,Third-party,Abuser,Abuser,None
1,104,Abuser,Abuser,Abuser,None
2,340,Victim,Abuser,Abuser,None
3,69,Victim,Third-party,Third-party,None
4,381,Abuser,Abuser,Abuser,None
5,391,Abuser,Abuser,Abuser,None
6,156,Victim,Third-party,Third-party\n\nThe narrator is commenting on t...,None
7,337,Victim,Abuser,Abuser,None
8,84,Third-party,Third-party,Third-party,None
9,177,Third-party,Abuser,Abuser,None


In [39]:
print("Valid labels:", zero_shot_test10["pred_label"].isin(LABELS).sum())
print("Errors:", zero_shot_test10["error"].notna().sum())
zero_shot_test10_eval

Valid labels: 10
Errors: 0


,setup,label,precision,recall,f1,support
0,zero_shot_test10_cerebras,Abuser,0.428571,1.000000,0.600000,3.0
1,zero_shot_test10_cerebras,Victim,0.000000,0.000000,0.000000,4.0
2,zero_shot_test10_cerebras,Third-party,0.333333,0.333333,0.333333,3.0
3,zero_shot_test10_cerebras,macro avg,0.253968,0.444444,0.311111,10.0
4,zero_shot_test10_cerebras,accuracy,0.400000,0.400000,0.400000,10.0


In [40]:
zero_shot_df, zero_shot_eval = run_and_save_setup(
    setup_name="zero_shot_full_cerebras",
    prompt_builder_fn=zero_shot_builder,
    cot=False,
    max_rows=None,
    save_every=5,
    notes="Task 3A zero-shot full test set"
)

zero_shot_eval

/tmp/ipykernel_2159/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


zero_shot_full_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_2159/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


,setup,label,precision,recall,f1,support
0,zero_shot_full_cerebras,Abuser,0.766990,0.711712,0.738318,111.0
1,zero_shot_full_cerebras,Victim,0.916667,0.183333,0.305556,60.0
2,zero_shot_full_cerebras,Third-party,0.188235,0.551724,0.280702,29.0
3,zero_shot_full_cerebras,macro avg,0.623964,0.482256,0.441525,200.0
4,zero_shot_full_cerebras,accuracy,0.530000,0.530000,0.530000,200.0


In [41]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Reply with only: Abuser"}],
    temperature=0,
    max_completion_tokens=16
)

print("prompt_tokens:", response.usage.prompt_tokens)
print("completion_tokens:", response.usage.completion_tokens)
print("total_tokens:", response.usage.total_tokens)

prompt_tokens: 41
completion_tokens: 16
total_tokens: 57


In [22]:
with zipfile.ZipFile("/content/task3_results_backup.zip", "r") as z:
    z.extractall("/content/task3_results")

zero_shot_df = pd.read_csv("/content/task3_results/zero_shot_full_cerebras.csv")
zero_shot_eval = pd.read_csv("/content/task3_results/eval/zero_shot_full_cerebras_eval.csv")

In [23]:
source_df[["sample_id", "role", "body_clean"]].sample(20, random_state=42)

,sample_id,role,body_clean
15,145,Victim,"This is a great app, but it is kinda like a st..."
9,34,Abuser,This app is great for tracking family!!! When ...
115,119,Victim,Well it is not very accurate all the time but ...
78,257,Abuser,Without them. I wouldn’t have caught my ex hus...
66,390,Abuser,Good app. I can spy cop talk!!!!
45,314,Third-party,"Everytime I turn the flash on, on the front ca..."
143,326,Victim,This app absolutely ruin my life. My girlfrien...
177,96,Abuser,Me and my friends use this app so we can stalk...
200,335,Abuser,It's a fun app.. But I miss looking at other p...
180,285,Abuser,I never do these lil app spy on # and so on bu...


In [30]:
HANDPICKED_SAMPLE_IDS = [193, 326, 131, 137, 82, 314]

handpicked_pool = source_df[source_df["sample_id"].isin(HANDPICKED_SAMPLE_IDS)].copy()
handpicked_pool[["sample_id", "role", "body_clean"]]

,sample_id,role,body_clean
18,193,Abuser,I love this app. I can use it to stalk my kids...
45,314,Third-party,"Everytime I turn the flash on, on the front ca..."
69,137,Abuser,Love stalking my family!
79,82,Victim,My mom is stalking me thanks to you guys XD
143,326,Victim,This app absolutely ruin my life. My girlfrien...
166,131,Third-party,Simple way to turn an old droid or iPhone into...


In [31]:
def get_handpicked_examples(k):
    ordered = handpicked_pool.set_index("sample_id").loc[HANDPICKED_SAMPLE_IDS].reset_index()
    return ordered.iloc[:k][["sample_id", "role", "body_clean"]].copy()

def make_handpicked_builder(k):
    examples_df = get_handpicked_examples(k)

    def _builder(row, pos, cot=False):
        return build_few_shot_prompt(row.body_clean, examples_df, cot=cot)

    return _builder

In [34]:
handpicked_results = {}
handpicked_evals = {}

for k in range(1, 7):
    df_k, eval_k = run_and_save_setup(
        setup_name=f"fewshot_handpicked_k{k}_cerebras",
        prompt_builder_fn=make_handpicked_builder(k),
        cot=False,
        max_rows=None,
        save_every=5,
        notes=f"Task 3A handpicked few-shot K={k}"
    )
    handpicked_results[k] = df_k
    handpicked_evals[k] = eval_k

Resuming fewshot_handpicked_k1_cerebras. Already completed: 200


/tmp/ipykernel_1652/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_handpicked_k1_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

Backup written to: /content/task3_results_backup.zip
Resuming fewshot_handpicked_k2_cerebras. Already completed: 200


/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


...

/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_handpicked_k6_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


In [22]:
def get_random_examples(k, seed=42):
    return source_df.sample(n=k, random_state=seed + k)[["sample_id", "role", "body_clean"]].copy()

def make_random_builder(k, seed=42):
    examples_df = get_random_examples(k, seed=seed)

    def _builder(row, pos, cot=False):
        return build_few_shot_prompt(row.body_clean, examples_df, cot=cot)

    return _builder

In [38]:
random_results = {}
random_evals = {}

for k in range(1, 6):
    df_k, eval_k = run_and_save_setup(
        setup_name=f"fewshot_random_k{k}_cerebras",
        prompt_builder_fn=make_random_builder(k, seed=42),
        cot=False,
        max_rows=None,
        save_every=5,
        notes=f"Task 3A random few-shot K={k}"
    )
    random_results[k] = df_k
    random_evals[k] = eval_k

Resuming fewshot_random_k1_cerebras. Already completed: 200


/tmp/ipykernel_1652/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_random_k1_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

Backup written to: /content/task3_results_backup.zip
Resuming fewshot_random_k2_cerebras. Already completed: 125


/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_1652/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


...

/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


Rate limited. Sleeping for 5 seconds...
Rate limited. Sleeping for 5 seconds...
Rate limited. Sleeping for 5 seconds...
Rate limited. Sleeping for 5 seconds...
Rate limited. Sleeping for 5 seconds...


/tmp/ipykernel_1652/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


Rate limited. Sleeping for 5 seconds...
Rate limited. Sleeping for 5 seconds...
Rate limited. Sleeping for 5 seconds...


KeyboardInterrupt: 

In [23]:
os.environ["CEREBRAS_API_KEY_2"] = userdata.get("CEREBRAS_API_KEY_2")

client = OpenAI(
    base_url="https://api.cerebras.ai/v1",
    api_key=os.environ["CEREBRAS_API_KEY_2"]
)

MODEL_NAME = "llama3.1-8b"
print("Cerebras key 2 loaded")
print("Model:", MODEL_NAME)

Cerebras key 2 loaded
Model: llama3.1-8b


In [24]:
df_k4, eval_k4 = run_and_save_setup(
    setup_name="fewshot_random_k4_cerebras",
    prompt_builder_fn=make_random_builder(4, seed=42),
    cot=False,
    max_rows=None,
    save_every=5,
    notes="Task 3A random few-shot K=4"
)

eval_k4

/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_random_k4_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


,setup,label,precision,recall,f1,support
0,fewshot_random_k4_cerebras,Abuser,0.701613,0.783784,0.740426,111.0
1,fewshot_random_k4_cerebras,Victim,0.888889,0.133333,0.231884,60.0
2,fewshot_random_k4_cerebras,Third-party,0.223881,0.517241,0.312500,29.0
3,fewshot_random_k4_cerebras,macro avg,0.604794,0.478119,0.428270,200.0
4,fewshot_random_k4_cerebras,accuracy,0.550000,0.550000,0.550000,200.0


In [25]:
df_k5, eval_k5 = run_and_save_setup(
    setup_name="fewshot_random_k5_cerebras",
    prompt_builder_fn=make_random_builder(5, seed=42),
    cot=False,
    max_rows=None,
    save_every=5,
    notes="Task 3A random few-shot K=5"
)

eval_k5

/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_random_k5_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


,setup,label,precision,recall,f1,support
0,fewshot_random_k5_cerebras,Abuser,0.768519,0.747748,0.757991,111.0
1,fewshot_random_k5_cerebras,Victim,0.928571,0.216667,0.351351,60.0
2,fewshot_random_k5_cerebras,Third-party,0.217949,0.586207,0.317757,29.0
3,fewshot_random_k5_cerebras,macro avg,0.638346,0.516874,0.475700,200.0
4,fewshot_random_k5_cerebras,accuracy,0.565000,0.565000,0.565000,200.0


In [26]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

source_matrix = vectorizer.fit_transform(source_df["body_clean"])
test_matrix = vectorizer.transform(test_df["body_clean"])
sim_matrix = cosine_similarity(test_matrix, source_matrix)

In [27]:
def get_similar_examples(test_row_position, k):
    sims = sim_matrix[test_row_position]
    top_idx = np.argsort(sims)[::-1][:k]
    return source_df.iloc[top_idx][["sample_id", "role", "body_clean"]].copy()

def make_similarity_builder(k):
    def _builder(row, pos, cot=False):
        examples_df = get_similar_examples(pos, k)
        return build_few_shot_prompt(row.body_clean, examples_df, cot=cot)
    return _builder

In [29]:
similar_results = {}
similar_evals = {}

for k in range(1, 6):
    df_k, eval_k = run_and_save_setup(
        setup_name=f"fewshot_similarity_k{k}_cerebras",
        prompt_builder_fn=make_similarity_builder(k),
        cot=False,
        max_rows=None,
        save_every=5,
        notes=f"Task 3A similarity few-shot K={k}"
    )
    similar_results[k] = df_k
    similar_evals[k] = eval_k

Resuming fewshot_similarity_k1_cerebras. Already completed: 200


/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_similarity_k1_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

Backup written to: /content/task3_results_backup.zip
Resuming fewshot_similarity_k2_cerebras. Already completed: 200


/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


...

/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_similarity_k5_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


In [30]:
backup_task3_outputs()

Backup written to: /content/task3_results_backup.zip


'/content/task3_results_backup.zip'

In [31]:
import zipfile, os, shutil

restore_dir = "/content/task3_results"

# optional: clear old restore dir first
if os.path.exists(restore_dir):
    shutil.rmtree(restore_dir)
os.makedirs(restore_dir, exist_ok=True)

backup_files = [
    "/content/task3_results_backup.zip",
    "/content/task3_results_backup (1).zip",
    "/content/task3_results_backup (2).zip",
]

for path in backup_files:
    if os.path.exists(path):
        print("Extracting:", path)
        with zipfile.ZipFile(path, "r") as z:
            z.extractall(restore_dir)

print("\nRestored files:")
for root, dirs, files in os.walk(restore_dir):
    for f in sorted(files):
        print(os.path.join(root, f))

Extracting: /content/task3_results_backup.zip
Extracting: /content/task3_results_backup (1).zip
Extracting: /content/task3_results_backup (2).zip

Restored files:
/content/task3_results/fewshot_handpicked_k1_cerebras.csv
/content/task3_results/fewshot_handpicked_k2_cerebras.csv
/content/task3_results/fewshot_handpicked_k3_cerebras.csv
/content/task3_results/fewshot_handpicked_k4_cerebras.csv
/content/task3_results/fewshot_handpicked_k5_cerebras.csv
/content/task3_results/fewshot_handpicked_k6_cerebras.csv
/content/task3_results/fewshot_random_k1_cerebras.csv
/content/task3_results/fewshot_random_k2_cerebras.csv
/content/task3_results/fewshot_random_k3_cerebras.csv
/content/task3_results/fewshot_random_k4_cerebras.csv
/content/task3_results/fewshot_random_k5_cerebras.csv
/content/task3_results/fewshot_similarity_k1_cerebras.csv
/content/task3_results/fewshot_similarity_k2_cerebras.csv
/content/task3_results/fewshot_similarity_k3_cerebras.csv
/content/task3_results/fewshot_similarity_k4_

In [32]:
for name in [
    "zero_shot_full_cerebras",
    "fewshot_handpicked_k1_cerebras",
    "fewshot_handpicked_k2_cerebras",
    "fewshot_handpicked_k3_cerebras",
    "fewshot_handpicked_k4_cerebras",
    "fewshot_handpicked_k5_cerebras",
    "fewshot_random_k1_cerebras",
    "fewshot_random_k2_cerebras",
    "fewshot_random_k3_cerebras",
    "fewshot_random_k4_cerebras",
    "fewshot_random_k5_cerebras",
    "fewshot_similarity_k1_cerebras",
    "fewshot_similarity_k2_cerebras",
    "fewshot_similarity_k3_cerebras",
    "fewshot_similarity_k4_cerebras",
    "fewshot_similarity_k5_cerebras",
]:
    csv_exists = os.path.exists(f"/content/task3_results/{name}.csv")
    eval_exists = os.path.exists(f"/content/task3_results/eval/{name}_eval.csv")
    print(name, "csv:", csv_exists, "| eval:", eval_exists)

zero_shot_full_cerebras csv: True | eval: True
fewshot_handpicked_k1_cerebras csv: True | eval: True
fewshot_handpicked_k2_cerebras csv: True | eval: True
fewshot_handpicked_k3_cerebras csv: True | eval: True
fewshot_handpicked_k4_cerebras csv: True | eval: True
fewshot_handpicked_k5_cerebras csv: True | eval: True
fewshot_random_k1_cerebras csv: True | eval: True
fewshot_random_k2_cerebras csv: True | eval: True
fewshot_random_k3_cerebras csv: True | eval: True
fewshot_random_k4_cerebras csv: True | eval: True
fewshot_random_k5_cerebras csv: True | eval: True
fewshot_similarity_k1_cerebras csv: True | eval: True
fewshot_similarity_k2_cerebras csv: True | eval: True
fewshot_similarity_k3_cerebras csv: True | eval: True
fewshot_similarity_k4_cerebras csv: True | eval: True
fewshot_similarity_k5_cerebras csv: True | eval: True


In [33]:
zip_path = shutil.make_archive("/content/task3_results_complete_backup", "zip", "/content/task3_results")
print(zip_path)

/content/task3_results_complete_backup.zip


In [34]:
zero_shot_eval = pd.read_csv("/content/task3_results/eval/zero_shot_full_cerebras_eval.csv")

handpicked_evals = {
    k: pd.read_csv(f"/content/task3_results/eval/fewshot_handpicked_k{k}_cerebras_eval.csv")
    for k in range(1, 6)
}

random_evals = {
    k: pd.read_csv(f"/content/task3_results/eval/fewshot_random_k{k}_cerebras_eval.csv")
    for k in range(1, 6)
}

similar_evals = {
    k: pd.read_csv(f"/content/task3_results/eval/fewshot_similarity_k{k}_cerebras_eval.csv")
    for k in range(1, 6)
}

all_eval_tables = [zero_shot_eval]

for k in range(1, 6):
    all_eval_tables.append(handpicked_evals[k])
    all_eval_tables.append(random_evals[k])
    all_eval_tables.append(similar_evals[k])

task3_eval_df = pd.concat(all_eval_tables, ignore_index=True)
task3_eval_df.to_csv("/content/task3_results/eval/task3_all_eval_metrics_cerebras.csv", index=False)

task3_eval_df.head(20)

,setup,label,precision,recall,f1,support
0,zero_shot_full_cerebras,Abuser,0.766990,0.711712,0.738318,111.0
1,zero_shot_full_cerebras,Victim,0.916667,0.183333,0.305556,60.0
2,zero_shot_full_cerebras,Third-party,0.188235,0.551724,0.280702,29.0
3,zero_shot_full_cerebras,macro avg,0.623964,0.482256,0.441525,200.0
4,zero_shot_full_cerebras,accuracy,0.530000,0.530000,0.530000,200.0
5,fewshot_handpicked_k1_cerebras,Abuser,0.860465,0.666667,0.751269,111.0
6,fewshot_handpicked_k1_cerebras,Victim,0.833333,0.333333,0.476190,60.0
7,fewshot_handpicked_k1_cerebras,Third-party,0.255556,0.793103,0.386555,29.0
8,fewshot_handpicked_k1_cerebras,macro avg,0.649785,0.597701,0.538005,200.0
9,fewshot_handpicked_k1_cerebras,accuracy,0.585000,0.585000,0.585000,200.0


In [35]:
LABELS = ["Abuser", "Victim", "Third-party"]

category_compare = task3_eval_df[task3_eval_df["label"].isin(LABELS)].copy()
category_compare = category_compare.sort_values(["label", "f1"], ascending=[True, False])

category_compare.head(30)

,setup,label,precision,recall,f1,support
75,fewshot_similarity_k5_cerebras,Abuser,0.762295,0.837838,0.798283,111.0
60,fewshot_similarity_k4_cerebras,Abuser,0.742188,0.855856,0.794979,111.0
25,fewshot_random_k2_cerebras,Abuser,0.754098,0.828829,0.789700,111.0
10,fewshot_random_k1_cerebras,Abuser,0.781818,0.774775,0.778281,111.0
30,fewshot_similarity_k2_cerebras,Abuser,0.725806,0.810811,0.765957,111.0
70,fewshot_random_k5_cerebras,Abuser,0.768519,0.747748,0.757991,111.0
15,fewshot_similarity_k1_cerebras,Abuser,0.741379,0.774775,0.757709,111.0
20,fewshot_handpicked_k2_cerebras,Abuser,0.890244,0.657658,0.756477,111.0
40,fewshot_random_k3_cerebras,Abuser,0.673759,0.855856,0.753968,111.0
65,fewshot_handpicked_k5_cerebras,Abuser,0.797980,0.711712,0.752381,111.0


In [36]:
macro_compare = task3_eval_df[task3_eval_df["label"] == "macro avg"].copy()
macro_compare = macro_compare.sort_values("f1", ascending=False)

macro_compare.head(10)

,setup,label,precision,recall,f1,support
78,fewshot_similarity_k5_cerebras,macro avg,0.646052,0.563187,0.538700,200.0
8,fewshot_handpicked_k1_cerebras,macro avg,0.649785,0.597701,0.538005,200.0
13,fewshot_random_k1_cerebras,macro avg,0.600350,0.553277,0.529161,200.0
28,fewshot_random_k2_cerebras,macro avg,0.597520,0.530874,0.524789,200.0
68,fewshot_handpicked_k5_cerebras,macro avg,0.646747,0.561184,0.523380,200.0
53,fewshot_handpicked_k4_cerebras,macro avg,0.623397,0.534058,0.505271,200.0
38,fewshot_handpicked_k3_cerebras,macro avg,0.617614,0.524599,0.497534,200.0
63,fewshot_similarity_k4_cerebras,macro avg,0.624004,0.529921,0.496828,200.0
73,fewshot_random_k5_cerebras,macro avg,0.638346,0.516874,0.475700,200.0
33,fewshot_similarity_k2_cerebras,macro avg,0.623702,0.497856,0.466357,200.0


In [37]:
zero_shot_df = pd.read_csv("/content/task3_results/zero_shot_full_cerebras.csv")

handpicked_results = {
    k: pd.read_csv(f"/content/task3_results/fewshot_handpicked_k{k}_cerebras.csv")
    for k in range(1, 6)
}

random_results = {
    k: pd.read_csv(f"/content/task3_results/fewshot_random_k{k}_cerebras.csv")
    for k in range(1, 6)
}

similar_results = {
    k: pd.read_csv(f"/content/task3_results/fewshot_similarity_k{k}_cerebras.csv")
    for k in range(1, 6)
}

In [38]:
def collect_qualitative_examples(result_df, test_df, n_correct=2, n_wrong=2):
    merged = result_df.merge(
        test_df[["sample_id", "body_clean"]],
        on="sample_id",
        how="left"
    )

    correct = merged[merged["true_label"] == merged["pred_label"]].head(n_correct)
    wrong = merged[merged["true_label"] != merged["pred_label"]].head(n_wrong)

    return pd.concat([correct, wrong], ignore_index=True)

In [39]:
qual_zero = collect_qualitative_examples(zero_shot_df, test_df)
qual_zero

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,104,Abuser,Abuser,Abuser,NaN,Great app to stalk your teenagers with! Love it!
1,381,Abuser,Abuser,Abuser,NaN,BEST APP EVER I CAN SPY ON EVERYBODY AND PRANKS
2,357,Third-party,Abuser,Abuser,NaN,I would like to have stealth mode where your n...
3,340,Victim,Abuser,Abuser,NaN,Love that your girlfriend or other crazy chick...


In [40]:
qual_handpicked_k3 = collect_qualitative_examples(handpicked_results[3], test_df)
qual_handpicked_k3

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,104,Abuser,Abuser,Abuser,NaN,Great app to stalk your teenagers with! Love it!
1,381,Abuser,Abuser,Abuser,NaN,BEST APP EVER I CAN SPY ON EVERYBODY AND PRANKS
2,357,Third-party,Abuser,Abuser,NaN,I would like to have stealth mode where your n...
3,340,Victim,Abuser,Abuser,NaN,Love that your girlfriend or other crazy chick...


In [ ]:
qual_random_k3 = collect_qualitative_examples(random_results[3], test_df)
qual_random_k3

In [41]:
qual_similarity_k3 = collect_qualitative_examples(similar_results[3], test_df)
qual_similarity_k3

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,104,Abuser,Abuser,Abuser,NaN,Great app to stalk your teenagers with! Love it!
1,381,Abuser,Abuser,Abuser,NaN,BEST APP EVER I CAN SPY ON EVERYBODY AND PRANKS
2,357,Third-party,Abuser,Abuser,NaN,I would like to have stealth mode where your n...
3,340,Victim,Abuser,Abuser,NaN,Love that your girlfriend or other crazy chick...


In [42]:
random_k3_df = pd.read_csv("/content/task3_results/fewshot_random_k3_cerebras.csv")
qual_random_k3 = collect_qualitative_examples(random_k3_df, test_df)
qual_random_k3

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,104,Abuser,Abuser,Abuser,NaN,Great app to stalk your teenagers with! Love it!
1,381,Abuser,Abuser,Abuser,NaN,BEST APP EVER I CAN SPY ON EVERYBODY AND PRANKS
2,357,Third-party,Abuser,Abuser,NaN,I would like to have stealth mode where your n...
3,340,Victim,Abuser,Abuser,NaN,Love that your girlfriend or other crazy chick...


In [43]:
backup_task3_outputs()

Backup written to: /content/task3_results_backup.zip


'/content/task3_results_backup.zip'

In [44]:
best_per_label = (
    task3_eval_df[task3_eval_df["label"].isin(LABELS)]
    .sort_values(["label", "f1"], ascending=[True, False])
    .groupby("label", as_index=False)
    .first()
)

best_per_label

,label,setup,precision,recall,f1,support
0,Abuser,fewshot_similarity_k5_cerebras,0.762295,0.837838,0.798283,111.0
1,Third-party,fewshot_handpicked_k1_cerebras,0.255556,0.793103,0.386555,29.0
2,Victim,fewshot_random_k2_cerebras,0.807692,0.350000,0.488372,60.0


In [45]:
best_macro = (
    task3_eval_df[task3_eval_df["label"] == "macro avg"]
    .sort_values("f1", ascending=False)
)

best_macro.head(10)

,setup,label,precision,recall,f1,support
78,fewshot_similarity_k5_cerebras,macro avg,0.646052,0.563187,0.538700,200.0
8,fewshot_handpicked_k1_cerebras,macro avg,0.649785,0.597701,0.538005,200.0
13,fewshot_random_k1_cerebras,macro avg,0.600350,0.553277,0.529161,200.0
28,fewshot_random_k2_cerebras,macro avg,0.597520,0.530874,0.524789,200.0
68,fewshot_handpicked_k5_cerebras,macro avg,0.646747,0.561184,0.523380,200.0
53,fewshot_handpicked_k4_cerebras,macro avg,0.623397,0.534058,0.505271,200.0
38,fewshot_handpicked_k3_cerebras,macro avg,0.617614,0.524599,0.497534,200.0
63,fewshot_similarity_k4_cerebras,macro avg,0.624004,0.529921,0.496828,200.0
73,fewshot_random_k5_cerebras,macro avg,0.638346,0.516874,0.475700,200.0
33,fewshot_similarity_k2_cerebras,macro avg,0.623702,0.497856,0.466357,200.0


In [46]:
best_per_label.to_csv("/content/task3_results/eval/task3_best_per_label.csv", index=False)
best_macro.to_csv("/content/task3_results/eval/task3_best_macro.csv", index=False)

In [47]:
best_setup_name = best_macro.iloc[0]["setup"]
best_setup_df = pd.read_csv(f"/content/task3_results/{best_setup_name}.csv")

qual_best_overall = collect_qualitative_examples(best_setup_df, test_df)
qual_best_overall

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,104,Abuser,Abuser,Abuser,NaN,Great app to stalk your teenagers with! Love it!
1,381,Abuser,Abuser,Abuser,NaN,BEST APP EVER I CAN SPY ON EVERYBODY AND PRANKS
2,357,Third-party,Abuser,Abuser,NaN,I would like to have stealth mode where your n...
3,340,Victim,Abuser,Abuser,NaN,Love that your girlfriend or other crazy chick...


In [48]:
backup_task3_outputs()

Backup written to: /content/task3_results_backup.zip


'/content/task3_results_backup.zip'

'''''''''''' Bonus Part '''''''''''''

In [49]:
best_macro = (
    task3_eval_df[task3_eval_df["label"] == "macro avg"]
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

best_macro.head(10)

,setup,label,precision,recall,f1,support
0,fewshot_similarity_k5_cerebras,macro avg,0.646052,0.563187,0.538700,200.0
1,fewshot_handpicked_k1_cerebras,macro avg,0.649785,0.597701,0.538005,200.0
2,fewshot_random_k1_cerebras,macro avg,0.600350,0.553277,0.529161,200.0
3,fewshot_random_k2_cerebras,macro avg,0.597520,0.530874,0.524789,200.0
4,fewshot_handpicked_k5_cerebras,macro avg,0.646747,0.561184,0.523380,200.0
5,fewshot_handpicked_k4_cerebras,macro avg,0.623397,0.534058,0.505271,200.0
6,fewshot_handpicked_k3_cerebras,macro avg,0.617614,0.524599,0.497534,200.0
7,fewshot_similarity_k4_cerebras,macro avg,0.624004,0.529921,0.496828,200.0
8,fewshot_random_k5_cerebras,macro avg,0.638346,0.516874,0.475700,200.0
9,fewshot_similarity_k2_cerebras,macro avg,0.623702,0.497856,0.466357,200.0


In [50]:
best_setup_name = best_macro.iloc[0]["setup"]
best_setup_name

'fewshot_similarity_k5_cerebras'

In [51]:
zero_shot_cot_df, zero_shot_cot_eval = run_and_save_setup(
    setup_name="zero_shot_cot_cerebras",
    prompt_builder_fn=zero_shot_builder,
    cot=True,
    max_rows=None,
    save_every=5,
    notes="Task 3 bonus CoT zero-shot"
)

zero_shot_cot_eval

/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


zero_shot_cot_cerebras:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


,setup,label,precision,recall,f1,support
0,zero_shot_cot_cerebras,Abuser,1.000000,0.75,0.857143,4.0
1,zero_shot_cot_cerebras,Victim,0.000000,0.00,0.000000,0.0
2,zero_shot_cot_cerebras,Third-party,0.000000,0.00,0.000000,0.0
3,zero_shot_cot_cerebras,macro avg,0.333333,0.25,0.285714,4.0
4,zero_shot_cot_cerebras,accuracy,0.750000,0.75,0.750000,4.0


In [52]:
def get_builder_from_setup_name(setup_name):
    if setup_name.startswith("fewshot_handpicked_k"):
        k = int(setup_name.split("_k")[1].split("_")[0])
        return make_handpicked_builder(k)
    elif setup_name.startswith("fewshot_random_k"):
        k = int(setup_name.split("_k")[1].split("_")[0])
        return make_random_builder(k, seed=42)
    elif setup_name.startswith("fewshot_similarity_k"):
        k = int(setup_name.split("_k")[1].split("_")[0])
        return make_similarity_builder(k)
    else:
        raise ValueError(f"Unsupported setup name: {setup_name}")

In [53]:
best_cot_builder = get_builder_from_setup_name(best_setup_name)
best_cot_setup_name = f"{best_setup_name}_cot"
best_cot_setup_name

'fewshot_similarity_k5_cerebras_cot'

In [54]:
best_fewshot_cot_df, best_fewshot_cot_eval = run_and_save_setup(
    setup_name=best_cot_setup_name,
    prompt_builder_fn=best_cot_builder,
    cot=True,
    max_rows=None,
    save_every=5,
    notes=f"Task 3 bonus CoT for best overall setup: {best_setup_name}"
)

best_fewshot_cot_eval

/tmp/ipykernel_5154/127617397.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


fewshot_similarity_k5_cerebras_cot:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fut

Backup written to: /content/task3_results_backup.zip


/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()
/tmp/ipykernel_5154/2790179092.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat()


,setup,label,precision,recall,f1,support
0,fewshot_similarity_k5_cerebras_cot,Abuser,1.000000,1.000000,1.000000,7.0
1,fewshot_similarity_k5_cerebras_cot,Victim,1.000000,1.000000,1.000000,3.0
2,fewshot_similarity_k5_cerebras_cot,Third-party,0.000000,0.000000,0.000000,0.0
3,fewshot_similarity_k5_cerebras_cot,macro avg,0.666667,0.666667,0.666667,10.0
4,fewshot_similarity_k5_cerebras_cot,accuracy,1.000000,1.000000,1.000000,10.0


In [55]:
base_zero_eval = pd.read_csv("/content/task3_results/eval/zero_shot_full_cerebras_eval.csv")
base_best_eval = pd.read_csv(f"/content/task3_results/eval/{best_setup_name}_eval.csv")

cot_compare = pd.concat([
    base_zero_eval.assign(version="zero_shot_non_cot"),
    zero_shot_cot_eval.assign(version="zero_shot_cot"),
    base_best_eval.assign(version=f"{best_setup_name}_non_cot"),
    best_fewshot_cot_eval.assign(version=f"{best_setup_name}_cot")
], ignore_index=True)

cot_compare

,setup,label,precision,recall,f1,support,version
0,zero_shot_full_cerebras,Abuser,0.766990,0.711712,0.738318,111.0,zero_shot_non_cot
1,zero_shot_full_cerebras,Victim,0.916667,0.183333,0.305556,60.0,zero_shot_non_cot
2,zero_shot_full_cerebras,Third-party,0.188235,0.551724,0.280702,29.0,zero_shot_non_cot
3,zero_shot_full_cerebras,macro avg,0.623964,0.482256,0.441525,200.0,zero_shot_non_cot
4,zero_shot_full_cerebras,accuracy,0.530000,0.530000,0.530000,200.0,zero_shot_non_cot
5,zero_shot_cot_cerebras,Abuser,1.000000,0.750000,0.857143,4.0,zero_shot_cot
6,zero_shot_cot_cerebras,Victim,0.000000,0.000000,0.000000,0.0,zero_shot_cot
7,zero_shot_cot_cerebras,Third-party,0.000000,0.000000,0.000000,0.0,zero_shot_cot
8,zero_shot_cot_cerebras,macro avg,0.333333,0.250000,0.285714,4.0,zero_shot_cot
9,zero_shot_cot_cerebras,accuracy,0.750000,0.750000,0.750000,4.0,zero_shot_cot


In [56]:
cot_compare.to_csv("/content/task3_results/eval/task3_cot_comparison.csv", index=False)

In [57]:
qual_zero_cot = collect_qualitative_examples(zero_shot_cot_df, test_df)
qual_best_cot = collect_qualitative_examples(best_fewshot_cot_df, test_df)

qual_zero_cot

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,228,Abuser,Abuser,Reason: The narrator endorses the app for spyi...,None,Awesome!!! Good for spying! Lol
1,359,Abuser,Abuser,Reason: The narrator endorses the app's spying...,None,Very nice and spy app .
2,357,Third-party,None,Reason: The narrator wants to use the app to s...,None,I would like to have stealth mode where your n...
3,104,Abuser,None,Reason: The narrator endorses using the app to...,None,Great app to stalk your teenagers with! Love it!


In [58]:
qual_best_cot

,sample_id,true_label,pred_label,raw_output,error,body_clean
0,95,Victim,Victim,Reason: The review is a repetition of Example ...,None,This app is a terrible app. All It is for is t...
1,266,Abuser,Abuser,Reason: The narrator endorses and personally u...,None,But overall a very good app to have. I can now...
2,357,Third-party,None,Reason: The narrator wants to anonymously trol...,None,I would like to have stealth mode where your n...
3,104,Abuser,None,Reason: The narrator endorses and personally u...,None,Great app to stalk your teenagers with! Love it!
